<a href="https://colab.research.google.com/github/artemprilubcenko2288/blank-app/blob/main/%D0%BF%D1%80%D0%BE%D0%B5%D0%BA%D1%82%D0%B8%D0%BA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive

drive.mount("/content/gdrive")

Mounted at /content/gdrive


In [4]:
from pathlib import Path
import json
import hashlib
from datetime import datetime

LIBRARY = Path("/content/gdrive/MyDrive/Научная библиотека")
PDF_FOLDER = LIBRARY / "PDF"
CATALOG_FILE = LIBRARY / "catalog.json"

PDF_FOLDER.mkdir(parents=True, exist_ok=True)

def load_catalog():
    if CATALOG_FILE.exists():
        return json.loads(CATALOG_FILE.read_text(encoding="utf-8"))
    return []

def save_catalog(catalog):
    temporary = CATALOG_FILE.with_suffix(".tmp")
    temporary.write_text(
        json.dumps(catalog, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    temporary.replace(CATALOG_FILE)

catalog = load_catalog()

print(f"Библиотека готова. Статей в каталоге: {len(catalog)}")

Библиотека готова. Статей в каталоге: 0


In [5]:
from google.colab import files

uploaded = files.upload()
catalog = load_catalog()

known_ids = {article["id"] for article in catalog}

for filename, content in uploaded.items():
    if not filename.lower().endswith(".pdf"):
        print(f"Пропущен файл не в формате PDF: {filename}")
        continue

    article_id = hashlib.sha256(content).hexdigest()

    if article_id in known_ids:
        print(f"Уже есть в библиотеке: {filename}")
        continue

    destination = PDF_FOLDER / f"{article_id}.pdf"
    destination.write_bytes(content)

    catalog.append({
        "id": article_id,
        "title": Path(filename).stem,
        "authors": "",
        "year": "",
        "tags": [],
        "note": "",
        "status": "Не прочитано",
        "original_filename": filename,
        "file": destination.name,
        "added_at": datetime.now().isoformat(timespec="seconds")
    })

    save_catalog(catalog)
    known_ids.add(article_id)

    print(f"Добавлена статья: {filename}")

print(f"\nВсего статей: {len(catalog)}")

Saving Institutional trust in authoritarian regimes  evidence from Central Asia.pdf to Institutional trust in authoritarian regimes  evidence from Central Asia.pdf
Saving TongfiKimetal-IS2018-Correspondence.pdf to TongfiKimetal-IS2018-Correspondence.pdf
Saving veenendaal2014.pdf to veenendaal2014.pdf
Saving military-service-and-immigrants-integration-evidence-from-the-vietnam-draft-lotteries.pdf to military-service-and-immigrants-integration-evidence-from-the-vietnam-draft-lotteries.pdf
Добавлена статья: Institutional trust in authoritarian regimes  evidence from Central Asia.pdf
Добавлена статья: TongfiKimetal-IS2018-Correspondence.pdf
Добавлена статья: veenendaal2014.pdf
Добавлена статья: military-service-and-immigrants-integration-evidence-from-the-vietnam-draft-lotteries.pdf

Всего статей: 4


In [6]:
import pandas as pd
from IPython.display import display

def show_library(query=""):
    catalog = load_catalog()
    rows = []

    for article in catalog:
        searchable = " ".join([
            article["title"],
            article["authors"],
            str(article["year"]),
            " ".join(article["tags"]),
            article["note"]
        ]).casefold()

        if query.casefold() in searchable:
            rows.append({
                "ID": article["id"][:12],
                "Название": article["title"],
                "Авторы": article["authors"],
                "Год": article["year"],
                "Теги": ", ".join(article["tags"]),
                "Статус": article["status"],
                "Заметка": article["note"]
            })

    if rows:
        display(pd.DataFrame(rows))
    else:
        print("Ничего не найдено.")

show_library()

,ID,Название,Авторы,Год,Теги,Статус,Заметка
0,8d751b67d7d9,Institutional trust in authoritarian regimes ...,,,,Не прочитано,
1,b76864007bf8,TongfiKimetal-IS2018-Correspondence,,,,Не прочитано,
2,917253a8983f,veenendaal2014,,,,Не прочитано,
3,3330f23e1922,military-service-and-immigrants-integration-ev...,,,,Не прочитано,


In [8]:
show_library("institutional")

,ID,Название,Авторы,Год,Теги,Статус,Заметка
0,8d751b67d7d9,Institutional trust in authoritarian regimes ...,,,,Не прочитано,


In [9]:
def edit_article(
    short_id,
    title=None,
    authors=None,
    year=None,
    tags=None,
    note=None,
    status=None
):
    catalog = load_catalog()
    matches = [
        article for article in catalog
        if article["id"].startswith(short_id)
    ]

    if len(matches) != 1:
        print("Нужен ID, соответствующий ровно одной статье.")
        return

    article = matches[0]

    updates = {
        "title": title,
        "authors": authors,
        "year": year,
        "tags": tags,
        "note": note,
        "status": status
    }

    for field, value in updates.items():
        if value is not None:
            article[field] = value

    save_catalog(catalog)
    print(f"Сохранено: {article['title']}")

In [10]:
edit_article(
    short_id="b76864007bf8",
    title="Correspondence",
    authors="no name",
    year="no",
    tags=["институты", "государственное управление"],
    note="Использовать для теоретической главы.",
    status="Читаю"
)

Сохранено: Correspondence


In [15]:
import ipywidgets as widgets
from IPython.display import display

def create_editor():
    articles = load_catalog()

    if not articles:
        print("Библиотека пуста. Сначала загрузи PDF.")
        return

    layout = widgets.Layout(width="95%")
    style = {"description_width": "100px"}

    selector = widgets.Dropdown(
        options=[
            (
                f"{article['title']} [{article['id'][:8]}]",
                article["id"]
            )
            for article in articles
        ],
        description="Статья:",
        layout=layout,
        style=style
    )

    title = widgets.Text(
        description="Название:",
        layout=layout,
        style=style
    )

    authors = widgets.Text(
        description="Авторы:",
        layout=layout,
        style=style
    )

    year = widgets.Text(
        description="Год:",
        layout=layout,
        style=style
    )

    tags = widgets.Text(
        description="Теги:",
        placeholder="политология, институты, для магистерской",
        layout=layout,
        style=style
    )

    note = widgets.Textarea(
        description="Заметка:",
        placeholder="О чём статья и зачем она тебе",
        layout=widgets.Layout(width="95%", height="140px"),
        style=style
    )

    statuses = ["Не прочитано", "Читаю", "Прочитано", "Использовать в работе"]

    for article in articles:
        saved_status = article.get("status")
        if saved_status and saved_status not in statuses:
            statuses.append(saved_status)

    status = widgets.Dropdown(
        options=statuses,
        description="Статус:",
        layout=layout,
        style=style
    )

    save_button = widgets.Button(
        description="Сохранить изменения",
        button_style="success",
        layout=widgets.Layout(width="230px")
    )

    message = widgets.Output()

    def fill_fields(change=None):
        message.clear_output()

        current = next(
            (
                article for article in load_catalog()
                if article["id"] == selector.value
            ),
            None
        )

        if current is None:
            with message:
                print("Статья не найдена. Запусти ячейку формы заново.")
            save_button.disabled = True
            return

        save_button.disabled = False
        title.value = current.get("title", "")
        authors.value = current.get("authors", "")
        year.value = str(current.get("year", ""))
        tags.value = ", ".join(current.get("tags", []))
        note.value = current.get("note", "")

        saved_status = current.get("status", "Не прочитано")

        if saved_status not in status.options:
            status.options = list(status.options) + [saved_status]

        status.value = saved_status

    def save_changes(button):
        message.clear_output()

        with message:
            if not title.value.strip():
                print("Введи название статьи.")
                return

            try:
                catalog = load_catalog()

                article = next(
                    (
                        item for item in catalog
                        if item["id"] == selector.value
                    ),
                    None
                )

                if article is None:
                    print("Статья не найдена. Запусти форму заново.")
                    return

                clean_tags = list(dict.fromkeys(
                    tag.strip()
                    for tag in tags.value.split(",")
                    if tag.strip()
                ))

                article.update({
                    "title": title.value.strip(),
                    "authors": authors.value.strip(),
                    "year": year.value.strip(),
                    "tags": clean_tags,
                    "note": note.value.strip(),
                    "status": status.value
                })

                save_catalog(catalog)

            except Exception as error:
                print(f"Не удалось сохранить: {error}")
                return

            print(f"Изменения сохранены: {article['title']}")

    selector.observe(fill_fields, names="value")
    save_button.on_click(save_changes)

    fill_fields()

    display(widgets.VBox([
        widgets.HTML("<h3>Редактирование карточки статьи</h3>"),
        selector,
        title,
        authors,
        year,
        tags,
        note,
        status,
        save_button,
        message
    ]))

create_editor()